## SEC Filings automation. 
For a given stock ticker (get_ticker_given_name)
1. Latest Financials 
	  1. Go to [EDGAR Filings Search](https://www.sec.gov/search-filings) (get_latest_filings)
	  2. Get financials from latest Annual (10-K) and Quarterly (10-Q) report filings (get_financial_statement)
  2. Stock Price 
	  1. Latest stock price (get_stock_price)
	  2. Analyst Ratings (get_analyst_rating_summary)
  3. Latest News (get_yahoo_news)


In [7]:
from edgar import *
import os
from collections.abc import Iterable
from typing import Literal
import pandas as pd 
from edgar.xbrl.stitching import XBRLS

In [8]:
# The stock ticker 
ticker = 'MU'

In [9]:
def ensure_list(item):
    """
    Ensures the input is returned as a list.
    - If input is None → returns an empty list.
    - If input is a string → wraps it in a list.
    - If input is a non-iterable → wraps it in a list.
    - If input is an iterable (excluding string/bytes) → converts it to a list.
    """
    if item is None:
        return []
    if isinstance(item, str) or not isinstance(item, Iterable):
        return [item]
    
    return list(item)

In [33]:

def reshape_financial_df(df: pd.DataFrame) -> pd.DataFrame:
    # Keep 'label' and 'concept' if present
    id_vars = ['label']
    if 'concept' in df.columns:
        id_vars.append('concept')

    long_df = df.melt(
        id_vars=id_vars,
        var_name="fiscal_date",
        value_name="amount"
    )
    sorted_df = long_df.sort_values(by=["label", "fiscal_date"])
    return sorted_df

In [10]:
def set_sec_client():
    """
    Initializes and returns the SEC client with identity set.

    Identity (email) is fetched from the SEC_IDENTITY environment variable.
    """
    email = os.getenv("SEC_IDENTITY", "default@example.com")
    set_identity(email)
    # You can optionally return a client or None if set_identity is global
    return True

In [16]:
# Get the latest filings for the stock ticker 

def get_latest_filings(ticker: str, form_type: Optional[str] = None, n: int = 5, as_text: bool = True) -> str:
    """
    Fetches the latest filings for a given ticker using SEC-API.
    If form_type is specified, filters by that form (e.g., '10-K', '8-K').

    Args:
        ticker (str): The stock ticker (e.g., "AAPL").
        form_type (Optional[str]): The form type to filter (e.g., "10-K", "10-Q", "8-K"). If None, returns all types.
        n (int): Number of filings to retrieve. Defaults to 5.
        as_text (bool): for string output, use True. Defaults to False which gives a Filing object 
    
    Returns:
        str: A newline-separated string of the latest filings.
    """
    set_sec_client()

    c = Company(ticker)
    
    if form_type:
        filings = c.get_filings(form=form_type).latest(n)
    else:
        filings = c.get_filings().latest(n)
    
    filings = ensure_list(filings)
    if as_text:
        return "\n".join(str(f) for f in filings)
    else: 
        return filings

In [18]:
x = get_latest_filings(ticker, form_type=None,  n=10, as_text=True)
print(x)

Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='4', filing_date='2026-04-02', accession_no='0002058769-26-000003')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='4', filing_date='2026-04-02', accession_no='0001218363-26-000002')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='144', filing_date='2026-04-01', accession_no='0001628280-26-022696')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='8-K', filing_date='2026-04-01', accession_no='0001104659-26-038249')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='SCHEDULE 13G/A', filing_date='2026-03-27', accession_no='0000102909-26-001910')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='8-K', filing_date='2026-03-25', accession_no='0001104659-26-034174')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='10-Q', filing_date='2026-03-19', accession_no='0000723125-26-000006')
Filing(company='MICRON TECHNOLOGY INC', cik=723125, form='8-K', filing_date='2026-03-18', accession_no=

In [36]:
def get_financial_statement(
    ticker: str, 
    form_type: Literal["10-K", "10-Q"],
    statement_type: Literal["cashflow", "balance_sheet", "income"], 
    n: int = 1
) -> pd.DataFrame:
    """
    Fetches financial statement(s) (cash flow, balance sheet, or income statement) 
    as a pandas DataFrame for a given company ticker and form type

    Args:
        ticker (str): Stock ticker symbol (e.g., "AAPL", "MSFT").
        form_type (str): Type of SEC filing to use. Options: "10-K"- Annual financial statements (more comprehensive),"10-Q": Quarterly financial statements (more recent, but limited scope)
        statement_type (str): One of "cashflow", "balance_sheet", or "income".
        n (int): Number of recent 10-K filings to retrieve. Defaults to 5.

    Returns:
        markdown table: With financial statement data -columns being:
            label (e.g. Revenue), fiscal date (e.g. 2025-08-31), $ amount (e.g. 1,230,621).
       
    Raises:
        ValueError: If the statement_type is invalid or if no filings/statements are found.
    """
    set_sec_client()

    c = Company(ticker)
    financials = Company(ticker).get_financials()
    
    
    # Select the statement based on the requested type
    if statement_type == "cashflow":
        stmt = financials.cashflow_statement()
    elif statement_type == "balance_sheet":
        stmt = financials.balance_sheet()  
    elif statement_type == "income":
        stmt = financials.income_statement()
    
    else:
        raise ValueError(f"Unsupported statement type: {statement_type}")

    stmnt = stmt.to_dataframe()
    #stmnt_lf = reshape_financial_df(stmnt)
    #stmnt_lf = stmnt_lf[["label", "fiscal_date", "amount"]]
    #stmnt_lf_md = stmnt_lf.to_markdown(index=False)
    stmnt_lf_md = stmnt
    return stmnt_lf_md

In [37]:
x = get_financial_statement(ticker, form_type="10-K", statement_type='income', n=1)
print(x)

                                              concept  \
0   us-gaap_RevenueFromContractWithCustomerExcludi...   
1   us-gaap_RevenueFromContractWithCustomerExcludi...   
2   us-gaap_RevenueFromContractWithCustomerExcludi...   
3   us-gaap_RevenueFromContractWithCustomerExcludi...   
4   us-gaap_RevenueFromContractWithCustomerExcludi...   
..                                                ...   
77                      us-gaap_EarningsPerShareBasic   
78                    us-gaap_EarningsPerShareDiluted   
79  us-gaap_WeightedAverageNumberOfSharesOutstandi...   
80  us-gaap_WeightedAverageNumberOfSharesOutstandi...   
81  us-gaap_WeightedAverageNumberOfDilutedSharesOu...   

                                              label  \
0                                           Revenue   
1                                              DRAM   
2                                              NAND   
3                             Other (primarily NOR)   
4                         Operating segm